In [0]:
# Note: To drop tables, run these commands in a separate SQL notebook or cell:
# DROP TABLE IF EXISTS workspace.gold.dim_producto_silver;
# DROP TABLE IF EXISTS workspace.gold.dim_vendedor_silver;
# DROP TABLE IF EXISTS workspace.gold.fact_venta_silver;
# DROP TABLE IF EXISTS workspace.gold.fact_ventas_final;

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="fact_ventas_final"
)
def fact_ventas_final():

    fact = spark.read.table(
        "workspace.silver.fact_venta"
    )

    producto = spark.read.table(
        "workspace.silver.dim_producto"
    ).drop("ing_date")

    vendedor = spark.read.table(
        "workspace.silver.dim_vendedor"
    ).drop("ing_date")

    return (
        fact
        .join(producto, "id_producto")
        .join(vendedor, "id_vendedor")
        .withColumn(
            "monto_total",
            col("cantidad") * col("precio_unitario")
        )
        .withColumn(
            "gold_load_timestamp",
            current_timestamp()
        )
    )